# **KELOMPOK:**


# 1.   Galuh Anggoro Wati (23.11.5751)
#2.   Anita Dewi Purwanti (23.11.5753)


#3.   Alifia Sansabila Sukardin (23.11.5785)
#4.   Sepvilia Fahturahma (23.11.5805)





# **Analisis dan Pemodelan Machine Learning untuk Prediksi Customer Churn**

In [ ]:
#  Install PySpark
!pip install pyspark -q

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Inisialisasi Spark
spark = SparkSession.builder.appName("ECommerceBigDataProject").getOrCreate()

Penjelasan:

pyspark → library utama untuk pemrosesan Big Data.

SparkSession → pintu masuk utama ke Spark.

functions (F) → fungsi bawaan Spark SQL (agregasi, transformasi).

DoubleType → mengatur tipe data numerik.

matplotlib & seaborn → visualisasi data.

pandas → konversi hasil Spark agar mudah divisualisasikan

## A. Load Data & Pemrosesan Batch (MapReduce RDD)

Memasukkan dataset e-commerce ke dalam sistem Spark.


##Penjelasan:

Memuat dua dataset e-commerce ke dalam Spark DataFrame.

header=true → baris pertama sebagai nama kolom.

sep=";" → pemisah kolom adalah titik koma.

In [ ]:
# Load Dataset
df_desc = spark.read.option("header", "true").option("sep", ";").csv("E Commerce Dataset pertama.csv")
df_main = spark.read.option("header", "true").option("sep", ";").csv("E Commerce Dataset kedua.csv")

# a. Pemrosesan batch dengan MapReduce pada RDD
# Menghitung jumlah kata unik dalam kolom 'Discerption' pada data deskripsi
description_rdd = df_desc.select("Discerption").rdd.flatMap(lambda x: x[0].split(" ")) \
                         .map(lambda word: (word, 1)) \
                         .reduceByKey(lambda a, b: a + b)

print("Hasil MapReduce (Word Count Deskripsi):", description_rdd.take(5))

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/E Commerce Dataset pertama.csv. SQLSTATE: 42K03

##Penjelasan:

select("Description") → mengambil kolom deskripsi produk.

rdd → mengubah DataFrame menjadi RDD.

flatMap() → memecah kalimat menjadi kata-kata.

map() → memberi nilai (kata, 1).

reduceByKey() → menghitung jumlah kemunculan tiap kata.
##Fungsi: menghitung frekuensi kata unik (Word Count).

## B. EDA (EXPLORATORY DATA ANALYSIS) & VISUALISASI

EDA dilakukan untuk memahami karakteristik dataset, seperti jumlah data, pola transaksi, dan distribusi nilai. Tahap ini membantu mengidentifikasi tren awal serta potensi permasalahan pada data sebelum masuk ke proses lanjutan.

In [ ]:
# Analisis distribusi Churn
churn_counts = df_main.groupBy("Churn").count().toPandas()

# Visualisasi (Sudah diperbaiki agar tidak muncul Future Warning)
plt.figure(figsize=(6, 4))
sns.barplot(data=churn_counts, x='Churn', y='count', hue='Churn', palette='viridis', legend=False)
plt.title('Distribusi Pelanggan (0: Stay, 1: Churn)')
plt.show()

##Penjelasan:

Mengelompokkan data berdasarkan status Churn.

Menghitung jumlah pelanggan Stay (0) dan Churn (1).

toPandas() → agar bisa divisualisasikan.


Menampilkan perbandingan jumlah pelanggan yang tetap dan yang churn dalam bentuk grafik batang.

In [ ]:
# Menghitung persentase agar lebih mudah diinterpretasikan
total_data = df_main.count()
churn_summary = df_main.groupBy("Churn").count().toPandas()
churn_summary['percentage'] = (churn_summary['count'] / total_data) * 100

print(churn_summary)

# Visualisasi Persentase
plt.figure(figsize=(6, 4))
sns.barplot(data=churn_summary, x='Churn', y='percentage', hue='Churn', palette='magma', legend=False)
plt.ylabel('Persentase (%)')
plt.title('Proporsi Churn Rate Pelanggan')
plt.show()

##Penjelasan:

Menghitung total data pelanggan.

Menghitung persentase pelanggan Stay dan Churn.

Membuat hasil lebih mudah diinterpretasikan dibanding angka absolut.

Menampilkan proporsi churn rate pelanggan dalam bentuk persentase untuk analisis bisnis.

In [ ]:
# 1. Menghitung persentase Churn berdasarkan status Complain
complain_churn_analysis = spark.sql("""
    SELECT
        Complain,
        Churn,
        COUNT(*) as Total_Pelanggan
    FROM ecommerce_main
    GROUP BY Complain, Churn
    ORDER BY Complain, Churn
""").toPandas()

# 2. Pivot data untuk mempermudah visualisasi (Membuat tabel perbandingan)
pivot_df = complain_churn_analysis.pivot(index='Complain', columns='Churn', values='Total_Pelanggan')
pivot_df.index = ['Tidak Komplain (0)', 'Komplain (1)']
pivot_df.columns = ['Stay (0)', 'Churn (1)']

print("Tabel Hubungan Complain vs Churn:")
print(pivot_df)

# 3. Visualisasi Stacked Bar Chart
plt.figure(figsize=(8, 6))
pivot_df.plot(kind='bar', stacked=True, color=['#4CAF50', '#F44336'])

plt.title('Proporsi Churn Berdasarkan Status Komplain')
plt.xlabel('Status Komplain Pelanggan')
plt.ylabel('Jumlah Pelanggan')
plt.xticks(rotation=0)
plt.legend(title='Status Churn')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 4. Menghitung Rasio Churn (Persentase)
print("\nAnalisis Rasio:")
df_ratio = spark.sql("""
    SELECT
        Complain,
        COUNT(*) as Total,
        SUM(Churn) as Total_Churn,
        ROUND((SUM(Churn) / COUNT(*)) * 100, 2) as Churn_Rate_Percentage
    FROM ecommerce_main
    GROUP BY Complain
""").show()

##Penjelasan:

Menggunakan Spark SQL untuk analisis relasi Complain dan Churn.

Menghitung jumlah pelanggan berdasarkan status komplain dan churn.

Hasil dikonversi ke Pandas untuk visualisasi.

##Pivot Data:
 [Mengubah data menjadi bentuk tabel silang agar mudah dibaca dan divisualisasikan.]

##Visualisasi Stacked Bar Chart:
1. Menampilkan perbandingan pelanggan Stay dan Churn berdasarkan status komplain.

2. Grafik stacked memudahkan melihat kontribusi churn akibat komplain.

## C. PREPROCESSING DATA (KUALITAS DATA)

In [ ]:
# 1. Casting Tipe Data ke Double
numeric_cols = ['Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp',
                'NumberOfDeviceRegistered', 'SatisfactionScore', 'NumberOfAddress',
                'Complain', 'OrderAmountHikeFromlastYear', 'CouponUsed',
                'OrderCount', 'DaySinceLastOrder', 'CashbackAmount', 'Churn']

for c in numeric_cols:
    df_main = df_main.withColumn(c, F.col(c).cast(DoubleType()))

# 2. Handling Missing Value (Imputasi dengan Mean)
for c in numeric_cols:
    mean_val = df_main.select(F.mean(F.col(c))).collect()[0][0]
    df_main = df_main.fillna({c: mean_val or 0.0}) # Isi dengan 0 jika mean null

print("C. Preprocessing Selesai: Casting tipe dan Handling Missing Value dilakukan.")


Mengubah seluruh kolom numerik ke tipe Double agar konsisten.

Menangani missing value dengan metode imputasi mean.

Tujuan: memastikan data bersih, konsisten, dan siap untuk analisis serta pemodelan ML.

## D. MANIPULASI DATA (SPARK SQL, CTE, AGREGASI)

In [ ]:
df_main.createOrReplaceTempView("ecommerce_main")
df_desc.createOrReplaceTempView("ecommerce_desc")

# Menggunakan CTE dan Subquery untuk melihat korelasi complain terhadap cashback
spark.sql("""
    WITH StatsTable AS (
        SELECT Complain, AVG(CashbackAmount) as AvgCashback, COUNT(*) as Total
        FROM ecommerce_main
        GROUP BY Complain
    )
    SELECT
        CASE WHEN Complain = 1 THEN 'Pernah Komplain' ELSE 'Tidak Pernah' END as Status,
        ROUND(AvgCashback, 2) as Rata_Rata_Cashback,
        Total
    FROM StatsTable
""").show()


DataFrame diubah menjadi temporary view untuk analisis SQL.

Menggunakan CTE untuk menghitung rata-rata cashback dan jumlah pelanggan berdasarkan status komplain.

Hasil menunjukkan rata-rata cashback hampir sama → cashback bukan faktor utama komplain.

## E. OPERASI PARTISI RDD (MAP, FLATMAP, BYKEY)

In [ ]:
# 1. Proses RDD (Map, Partition, Reduce)
rdd_category = df_main.rdd.map(lambda x: (x['PreferedOrderCat'], 1)) \
                         .partitionBy(2) \
                         .reduceByKey(lambda a, b: a + b)

# 2. Ubah RDD menjadi Spark DataFrame
df_rdd_result = rdd_category.toDF(["Kategori_Order", "Total_Order"])

print("Total Order per Kategori (via RDD):")
df_rdd_result.show()



Menggunakan RDD untuk menghitung total order per kategori produk.

Proses: map → reduceByKey → konversi ke DataFrame.

Kategori Mobile Phone memiliki jumlah order tertinggi.

## F. PERMODELAN ML (MLLIB) - KOMPARASI 2 ALGORITMA

In [ ]:
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Feature Engineering
indexer = StringIndexer(inputCol="PreferredLoginDevice", outputCol="DeviceIndex").setHandleInvalid("keep")
df_ml = indexer.fit(df_main).transform(df_main)

# Pilih fitur yang relevan
feature_cols = ['Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp', 'Complain', 'CashbackAmount']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
final_data = assembler.transform(df_ml).select("features", F.col("Churn").alias("label"))

# Split Data
train, test = final_data.randomSplit([0.8, 0.2], seed=42)

# Model 1: Logistic Regression
lr = LogisticRegression(labelCol="label", featuresCol="features")
model_lr = lr.fit(train)

# Model 2: Random Forest
rf = RandomForestClassifier(labelCol="label", featuresCol="features")
model_rf = rf.fit(train)

# Evaluasi Awal (Akurasi)
eval_acc = MulticlassClassificationEvaluator(metricName="accuracy")
print(f"Akurasi Logistic Regression: {eval_acc.evaluate(model_lr.transform(test)):.4f}")
print(f"Akurasi Random Forest: {eval_acc.evaluate(model_rf.transform(test)):.4f}")

Melakukan feature engineering (StringIndexer & VectorAssembler).

Data dibagi menjadi training (80%) dan testing (20%).

Membandingkan dua model klasifikasi:

Logistic Regression

Random Forest

Evaluasi awal menggunakan akurasi.

## G. HYPERPARAMETER TUNING & EVALUASI MODEL (RMSE, MSE, F1, DLL)

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Tuning pada model terbaik (Random Forest)
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .build()

cv = CrossValidator(estimator=rf, estimatorParamMaps=paramGrid,
                    evaluator=eval_acc, numFolds=3)
best_model = cv.fit(train)
predictions = best_model.transform(test)

# Evaluasi Metrik Lengkap
f1 = eval_acc.evaluate(predictions, {eval_acc.metricName: "f1"})
precision = eval_acc.evaluate(predictions, {eval_acc.metricName: "weightedPrecision"})
recall = eval_acc.evaluate(predictions, {eval_acc.metricName: "weightedRecall"})

# Interpretasi: Dalam klasifikasi Churn, Recall sangat penting agar kita tidak melewatkan pelanggan yang akan pergi.
print("\n--- HASIL EVALUASI FINAL (BEST MODEL) ---")
print(f"Accuracy  : {eval_acc.evaluate(predictions):.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

Melakukan hyperparameter tuning pada model Random Forest dengan Cross Validation.

Parameter yang diuji: jumlah pohon dan kedalaman pohon.

Evaluasi akhir menggunakan metrik:

Accuracy

F1-Score

Precision

Recall

Fokus pada Recall untuk mendeteksi pelanggan yang berpotensi churn.

In [ ]:
print(f"Jumlah partisi saat ini: {optimized_rdd.getNumPartitions()}")

## Hasil Evaluasi (Tambahan)

**Permodelan Algoritma (MLlib)
menyiapkan fitur numerik dan kategorikal, lalu melatih dua model berbeda.**

In [ ]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# 1. Memilih fitur hasil preprocessing sebelumnya
# Menggunakan fitur yang memiliki pengaruh besar terhadap Churn
feature_cols = ['Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp',
                'SatisfactionScore', 'Complain', 'OrderCount', 'CashbackAmount']

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data_final = assembler.transform(df_main).select("features", F.col("Churn").alias("label"))

# 2. Split Data (80% Training, 20% Testing)
train_data, test_data = data_final.randomSplit([0.8, 0.2], seed=42)

# 3. Komparasi 2 Algoritma
# Algoritma A: Logistic Regression
lr = LogisticRegression(labelCol="label", featuresCol="features")
model_lr = lr.fit(train_data)

# Algoritma B: Random Forest
rf = RandomForestClassifier(labelCol="label", featuresCol="features")
model_rf = rf.fit(train_data)

print("Permodelan Selesai. Siap untuk tahap evaluasi.")

Menyiapkan model machine learning di PySpark untuk memprediksi customer churn dengan memilih fitur penting, menggabungkannya ke dalam vektor fitur, mengubah target Churn menjadi label, lalu membagi data menjadi train dan test. Setelah itu, dua algoritma Logistic Regression dan Random Forest dilatih sehingga menghasilkan dua model yang siap dievaluasi performanya.

**Hyperparameter Tuning (Best Model)
mengoptimalkan Random Forest menggunakan CrossValidator.**

In [ ]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Membuat Grid Parameter untuk tuning Random Forest
paramGrid = (ParamGridBuilder()
             .addGrid(rf.numTrees, [10, 20, 50])
             .addGrid(rf.maxDepth, [5, 10])
             .build())

evaluator = MulticlassClassificationEvaluator(labelCol="Churn", metricName="accuracy")

# Cross Validation (3-Fold)
cv = CrossValidator(estimator=rf,
                    estimatorParamMaps=paramGrid,
                    evaluator=evaluator,
                    numFolds=3)

# Transformasi data test untuk evaluasi akhir
cv_model = cv.fit(assembler.transform(train_data))
best_rf_model = cv_model.bestModel

print("Hyperparameter Tuning Selesai. Model terbaik telah dipilih.")

Data latih dibagi menjadi 3 bagian untuk menguji berbagai kombinasi parameter secara bergantian. Kemudian melakukan hyperparameter tuning untuk model Random Forest agar performanya lebih optimal. Beberapa kombinasi parameter dicoba, yaitu jumlah pohon (numTrees: 10, 20, 50) dan kedalaman pohon (maxDepth: 5, 10) menggunakan ParamGridBuilder. Lalu digunakan CrossValidator dengan 3-fold cross validation.Model dinilai berdasarkan accuracy, dan setelah proses ini selesai, sistem otomatis memilih kombinasi parameter terbaik yang menghasilkan model Random Forest paling optimal

**Evaluasi Model & Visualisasi (Confusion Matrix)
menghitung metrik utama seperti Akurasi, Presisi, Recall, dan F1-Score, serta menampilkan Confusion Matrix.**

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import pandas as pd

# 1. Melakukan prediksi pada data test
predictions = best_model.transform(test_data)

# 2. Menghitung Metrik Evaluasi
acc = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
f1 = evaluator.evaluate(predictions, {evaluator.metricName: "f1"})
prec = evaluator.evaluate(predictions, {evaluator.metricName: "weightedPrecision"})
rec = evaluator.evaluate(predictions, {evaluator.metricName: "weightedRecall"})

print(f"--- METRIK EVALUASI ---")
print(f"Accuracy  : {acc:.4f}")
print(f"F1-Score  : {f1:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")

# 3. Visualisasi Confusion Matrix
y_true = predictions.select("label").toPandas()
y_pred = predictions.select("prediction").toPandas()
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Stay (0)', 'Churn (1)'],
            yticklabels=['Stay (0)', 'Churn (1)'])
plt.title('Confusion Matrix - Prediksi Churn')
plt.xlabel('Prediksi')
plt.ylabel('Aktual')
plt.show()

Tahap evaluasi akhir model churn. Model terbaik (best_model) digunakan untuk memprediksi data uji, lalu dihitung metrik performa utama yaitu accuracy (ketepatan keseluruhan), F1-score (keseimbangan precision & recall), precision (ketepatan prediksi churn), dan recall (kemampuan menangkap pelanggan yang benar-benar churn). Setelah itu dibuat confusion matrix untuk melihat detail hasil prediksi berapa banyak pelanggan yang benar diprediksi stay atau churn dan berapa yang salah yang divisualisasikan dalam bentuk heatmap agar lebih mudah dianalisis.

**ANALISIS FEATURE**

In [ ]:
# 1. Ekstrak nilai importance dari model terbaik
importances = best_model.featureImportances

# 2. Mapping ke nama kolom fitur
feature_list = ['Tenure', 'CityTier', 'WarehouseToHome', 'HourSpendOnApp',
                'SatisfactionScore', 'Complain', 'OrderCount', 'CashbackAmount']

# Buat DataFrame untuk visualisasi
fi_df = pd.DataFrame({
    'Feature': feature_list,
    'Importance': importances.toArray()
}).sort_values(by='Importance', ascending=False)

# 3. Visualisasi
plt.figure(figsize=(10, 6))
sns.barplot(data=fi_df, x='Importance', y='Feature', hue='Feature', palette='magma', legend=False)
plt.title('Fitur Paling Berpengaruh Terhadap Churn Pelanggan')
plt.xlabel('Skor Kepentingan (Importance Score)')
plt.ylabel('Fitur')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.show()

# Menampilkan angka pastinya
print("Daftar Fitur dan Skor Kepentingan:")
print(fi_df)

grafik menunjukkan bahwa Tenure adalah faktor paling berpengaruh terhadap churn, artinya lama pelanggan bergabung sangat menentukan apakah mereka akan tetap bertahan atau tidak. Setelah itu, CashbackAmount dan WarehouseToHome juga punya pengaruh besar, menandakan insentif finansial dan faktor logistik (jarak pengiriman) ikut memengaruhi keputusan pelanggan. Complain, OrderCount, dan SatisfactionScore memberi dampak menengah, sedangkan HourSpendOnApp dan CityTier pengaruhnya paling kecil. Secara umum, churn paling dipicu oleh kombinasi loyalitas pelanggan, kualitas layanan, dan manfaat yang diterima pelanggan.